In [1]:
from google.colab import userdata
import os

os.environ["AWS_ACCESS_KEY_ID"] = userdata.get('AWS_ACCESS_KEY_ID')
os.environ["AWS_SECRET_ACCESS_KEY"] = userdata.get('AWS_SECRET_ACCESS_KEY')
os.environ["AWS_REGION"] = userdata.get('AWS_REGION')

In [ ]:
!pip install langgraph langchain-aws -q

In [10]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_aws import ChatBedrockConverse

# Shared state
class AgentState(TypedDict):
    question: str
    research: str
    answer: str


# LLM
llm = ChatBedrockConverse(model="amazon.nova-lite-v1:0", region_name="us-east-1")


# -------------------------
# Agent 1 : Research Agent
# -------------------------
def research_agent(state: AgentState):

    # Get user question
    question = state["question"]

    # Ask LLM to research
    response = llm.invoke(
        f"Give short research on: {question}"
    )

    # Update state
    return {
        "research": response.content
    }


# -------------------------
# Agent 2 : Writer Agent
# -------------------------
def writer_agent(state: AgentState):

    research = state["research"]

    # Generate final answer
    response = llm.invoke(
        f"Write final answer using:\n{research}"
    )

    return {
        "answer": response.content
    }


# -------------------------
# Build Graph
# -------------------------
graph = StateGraph(AgentState)

# Add agents
graph.add_node("research", research_agent)
graph.add_node("writer", writer_agent)

# Flow
graph.add_edge(START, "research")
graph.add_edge("research", "writer")
graph.add_edge("writer", END)

# Compile
app = graph.compile()


# -------------------------
# Run Graph
# -------------------------
result = app.invoke({
    "question": "What is Generative AI?"
})

print(result["answer"])

**Generative AI**

**Definition:**
Generative AI refers to a class of artificial intelligence models that can generate new content, such as images, text, music, or other forms of media, by learning from existing data. These models can create outputs that resemble the training data but are not exact copies.

**Key Concepts:**

1. **Training Data:**
   - Generative models are trained on large datasets, learning patterns and structures within the data.

2. **Types of Generative Models:**
   - **Generative Adversarial Networks (GANs):** Consist of two neural networks (generator and discriminator) that compete against each other.
   - **Variational Autoencoders (VAEs):** Use a probabilistic approach to generate new data.
   - **Transformers:** Often used for generating text, leveraging deep learning techniques.

3. **Applications:**
   - **Art and Design:** Creating images and graphics.
   - **Music and Literature:** Composing music and generating text.
   - **Data Augmentation:** Enhancing